[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/08_dictionaries.ipynb)

*Python from Scratch · Notebook 8 of 12*

# Dictionaries

Key → value maps: lookups, updates, counting, grouping, nesting and dict comprehensions.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A dictionary is a phone book that jumps straight to the name instead of reading every page.

**What interviewers really test here:** Counting and grouping patterns, hash collisions, and turning O(n²) into O(n).

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Keys and values *(Basics)*
- [ ] 2. Reading safely and removing *(Basics)*
- [ ] 3. Looping over a dictionary *(Core)*
- [ ] 4. Counting and grouping *(Core)*
- [ ] 5. Dict comprehensions and sorting *(Advanced)*
- [ ] 6. Nested data and mappings *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **dictionary** (`dict`) is a **mutable mapping** from unique, hashable **keys** to **values**. It is Python's built-in **hash table** (associative array): you look values up by key rather than by position.



Properties:




- Keys are unique — assigning to an existing key replaces the value.
- Keys must be hashable (str, int, float, tuple of hashables, frozenset …); values can be any object.
- Insertion-ordered since Python 3.7 (iteration follows insertion order).
- Average O(1) lookup, insert and delete.

### Creating dictionaries

| Way | Example |
| --- | --- |
| Literal | `{"a": 1, "b": 2}`, `{}` |
| Constructor with keywords | `dict(a=1, b=2)` |
| From pairs | `dict([("a", 1), ("b", 2)])`, `dict(zip(keys, values))` |
| Comprehension | `{k: v for k, v in pairs}` |
| Same value for many keys | `dict.fromkeys(["a", "b"], 0)` |

### Methods and operations

| Operation | Description | Time |
| --- | --- | --- |
| `d[k]` | get value; KeyError if missing | O(1) |
| `d[k] = v` | add or update | O(1)\* |
| `del d[k]` | remove; KeyError if missing | O(1) |
| `k in d` | key membership | O(1) |
| `get(k, default=None)` | value or default | O(1) |
| `setdefault(k, default)` | get value; insert default first if missing | O(1) |
| `pop(k[, default])` | remove and return value | O(1) |
| `popitem()` | remove and return the last inserted pair | O(1) |
| `update(other)`, `\|=` | merge other into d | O(len other) |
| `a \| b` | new merged dict (3.9+) | O(len a + len b) |
| `keys() / values() / items()` | dynamic views | O(1) to create |
| `copy()` | shallow copy | O(n) |
| `clear()` | remove everything | O(n) |



\* amortised; occasionally the table is resized.

### How a dict works inside

1. Python computes `hash(key)`.
2. The hash picks a slot in an internal table.
3. If the slot holds a different key (a **collision**), Python probes other slots (open addressing).
4. When the table gets about two-thirds full, it grows and every key is re-inserted.



This is why keys must be hashable and why equal keys must hash equally: `1`, `1.0` and `True` are the **same** key.

### The collections helpers

| Class | What it adds | Example |
| --- | --- | --- |
| `Counter` | counts hashable items; `most_common(n)`; arithmetic between counters | `Counter("banana")` → {'a': 3, 'n': 2, 'b': 1} |
| `defaultdict(factory)` | missing keys get `factory()` automatically | `defaultdict(list)`, `defaultdict(int)` |
| `OrderedDict` | extra order operations (`move_to_end`) | LRU caches |
| `ChainMap` | search several dicts as one | layered config |

### Dict vs list vs set, and applications

|   | list | set | dict |
| --- | --- | --- | --- |
| Access by | position | — (membership only) | key |
| Lookup cost | O(n) by value | O(1) | O(1) by key |
| Stores | values | unique values | key → value pairs |



#### Applications




- Counting and frequency tables; grouping records.
- Caches and memoisation (remember computed results).
- Configuration, JSON data and API responses.
- Lookup tables, translations, phone books, inventories.
- Representing objects and graphs (adjacency lists).

---
# Part 2: Lessons, step by step

## Lesson 1: Keys and values

*Level: Basics*

A list finds items by **position**. A **dictionary** (`dict`) finds them by a **key** you choose — like a real dictionary maps a word to its meaning.



```python
phone = {"ana": "98450", "ben": "99001"}
phone["ana"]          # "98450"
phone["cy"] = "90000" # add a new pair
phone["ana"] = "11111" # update an existing key
len(phone)            # 3
"ben" in phone        # True — checks KEYS, not values
```



Each entry is a **key: value** pair. Keys are unique (assigning to an existing key replaces its value) and must be unchangeable types like strings, numbers or tuples. Values can be anything.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def dict_basics_demo():
    phone = {"ana": "98450", "ben": "99001"}
    phone["cy"] = "90000"
    phone["ana"] = "11111"
    has_ben = "ben" in phone
    print(phone["ana"], len(phone), has_ben)


dict_basics_demo()

11111 3 True


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 2: Reading safely and removing

*Level: Basics*

Reading a key that doesn't exist raises `KeyError`. Safer options:



| Code | If the key is missing |
| --- | --- |
| `d[key]` | KeyError |
| `d.get(key)` | returns None |
| `d.get(key, default)` | returns default |
| `if key in d:` | check first |



Removing: `del d[key]` (KeyError if missing), `d.pop(key, default)` (removes and returns the value), `d.clear()`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def get_demo():
    stock = {"pen": 10, "ink": 0}
    pens = stock.get("pen", 0)
    caps = stock.get("cap", 0)
    removed = stock.pop("ink", None)
    missing = stock.pop("glue", "not there")
    print(pens, caps, removed, missing, stock)


get_demo()

10 0 0 not there {'pen': 10}


**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 3: Looping over a dictionary

*Level: Core*

```python
for name in marks:                 # keys
for m in marks.values():           # values
for name, m in marks.items():      # (key, value) pairs — most useful
```



`keys()`, `values()` and `items()` return live **views** that reflect later changes. Since Python 3.7, dicts keep keys in the order they were inserted.



**⚠️ Watch out:** Don't add or delete keys while looping over a dict — loop over `list(d)` instead.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def loop_dict_demo():
    marks = {"ana": 72, "ben": 35, "cy": 88}
    total = 0
    for name, m in marks.items():
        status = "pass" if m >= 40 else "fail"
        print(name, m, status)
        total += m
    print("average", round(total / len(marks), 1))


loop_dict_demo()

ana 72 pass
ben 35 fail
cy 88 pass
average 65.0


**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q10](#q10), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 4: Counting and grouping

*Level: Core*

Two patterns cover most dictionary problems:



#### Counting



```python
counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1
```



#### Grouping



```python
groups = {}
for w in words:
    groups.setdefault(w[0], []).append(w)
```



The `collections` module makes them even shorter: `Counter(words)` counts, and `defaultdict(list)` creates an empty list the first time a key is used.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def counting_demo():
    words = ["red", "blue", "red", "green", "blue", "red"]
    counts = {}
    for w in words:
        counts[w] = counts.get(w, 0) + 1
    by_length = {}
    for w in words:
        by_length.setdefault(len(w), []).append(w)
    print(counts)
    print(by_length)


counting_demo()

{'red': 3, 'blue': 2, 'green': 1}
{3: ['red', 'red', 'red'], 4: ['blue', 'blue'], 5: ['green']}


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q15](#q15), [Q16](#q16), [Q7](#q7), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 5: Dict comprehensions and sorting

*Level: Advanced*

```python
squares = {n: n * n for n in range(5)}
passed = {name: m for name, m in marks.items() if m >= 40}
inverted = {v: k for k, v in d.items()}
```



Sorting a dict means sorting its keys or items:



```python
sorted(d)                              # keys alphabetically
sorted(d, key=d.get, reverse=True)     # keys by value, highest first
sorted(d.items(), key=lambda kv: kv[1])# (key, value) pairs by value
max(d, key=d.get)                      # key with the largest value
```



Merge dicts with `a | b` (Python 3.9+) or `{**a, **b}`; the right-hand side wins on shared keys.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def comprehension_demo():
    marks = {"ana": 72, "ben": 35, "cy": 88}
    passed = {n: m for n, m in marks.items() if m >= 40}
    ranking = sorted(marks, key=marks.get, reverse=True)
    best = max(marks, key=marks.get)
    merged = marks | {"dia": 91}
    print(passed, ranking, best, merged)


comprehension_demo()

{'ana': 72, 'cy': 88} ['cy', 'ana', 'ben'] cy {'ana': 72, 'ben': 35, 'cy': 88, 'dia': 91}


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q9](#q9), [Q17](#q17), [Q14](#q14), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 6: Nested data and mappings

*Level: Advanced*

Real data is often a dict of dicts or a list of dicts (this is what JSON looks like):



```python
students = {
    "ana": {"age": 21, "skills": ["python", "sql"]},
    "ben": {"age": 25, "skills": ["java"]},
}
students["ana"]["skills"][0]                  # "python"
students.get("zed", {}).get("age")            # None, no error
```



Dicts are also perfect as **lookup tables** (Roman numerals, prices) and for checking **one-to-one mappings** between two sequences (word patterns, isomorphic strings).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def nested_demo():
    students = {"ana": {"age": 21, "skills": ["python", "sql"]}, "ben": {"age": 25, "skills": ["java"]}}
    first_skill = students["ana"]["skills"][0]
    zed_age = students.get("zed", {}).get("age")
    students["ben"]["skills"].append("python")
    pythonistas = [name for name, info in students.items() if "python" in info["skills"]]
    print(first_skill, zed_age, pythonistas)


nested_demo()

python None ['ana', 'ben']


**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q20](#q20), [Q12](#q12). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Word Frequency

🟢 Warm-up · *Counting with a dict*

Return a dict mapping each word (lower-cased) to how many times it appears.

**Examples**

- `text = 'the cat the hat'` → `{'the': 2, 'cat': 1, 'hat': 1}`
- `text = ''` → `{}`

In [ ]:
def word_freq(text):
    """
    Return a dict mapping each word (lower-cased) to how many times it appears.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(word_freq, [
    (('the cat the hat',), {'the': 2, 'cat': 1, 'hat': 1}),
    (('',), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each word, count it again with `list.count` (scans the whole list each time).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One pass: `counts[w] = counts.get(w, 0) + 1`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each word, count it again with `list.count` (scans the whole list each time).

```python
def word_freq(text):
    words = text.lower().split()
    result = {}
    for w in words:
        result[w] = words.count(w)
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One pass: `counts[w] = counts.get(w, 0) + 1`.

```python
def word_freq(text):
    counts = {}
    for w in text.lower().split():
        counts[w] = counts.get(w, 0) + 1
    return counts
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each word, count it again with `list.count` (scans the whole list each time). That costs O(n²) time and O(n) space."

"I can do better: One pass: `counts[w] = counts.get(w, 0) + 1`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/word-frequency)

<a id="q2"></a>
### Q2. Character Frequency

🟢 Warm-up · *Counter*

Return a dict of character → count for the letters of `word`.

**Examples**

- `word = 'hello'` → `{'h': 1, 'e': 1, 'l': 2, 'o': 1}`
- `word = ''` → `{}`

In [ ]:
def char_freq(word):
    """
    Return a dict of character → count for the letters of word.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(char_freq, [
    (('hello',), {'h': 1, 'e': 1, 'l': 2, 'o': 1}),
    (('',), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check whether the key exists with `in` and branch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`collections.Counter` does the counting.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check whether the key exists with `in` and branch.

```python
def char_freq(word):
    counts = {}
    for ch in word:
        if ch in counts:
            counts[ch] = counts[ch] + 1
        else:
            counts[ch] = 1
    return counts
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `collections.Counter` does the counting.

```python
def char_freq(word):
    from collections import Counter
    return dict(Counter(word))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check whether the key exists with `in` and branch. That costs O(n) time and O(k) space."

"I can do better: `collections.Counter` does the counting. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/character-frequency)

<a id="q3"></a>
### Q3. Invert a Dictionary

🟢 Warm-up · *Dict comprehension*

Swap keys and values (all values are unique).

**Examples**

- `d = {'a': 1, 'b': 2}` → `{1: 'a', 2: 'b'}`
- `d = {}` → `{}`

In [ ]:
def invert(d):
    """
    Swap keys and values (all values are unique).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(invert, [
    (({'a': 1, 'b': 2},), {1: 'a', 2: 'b'}),
    (({},), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over keys and look each value up.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`{v: k for k, v in d.items()}`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over keys and look each value up.

```python
def invert(d):
    result = {}
    for key in d:
        value = d[key]
        result[value] = key
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `{v: k for k, v in d.items()}`.

```python
def invert(d):
    return {v: k for k, v in d.items()}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over keys and look each value up. That costs O(n) time and O(n) space."

"I can do better: `{v: k for k, v in d.items()}`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/invert-a-dictionary)

<a id="q4"></a>
### Q4. Safe Lookup

🟢 Warm-up · *get() with a default*

Return the price of `item` from the price dict, or 0 if it isn't there.

**Examples**

- `prices = {'pen': 10, 'ink': 25}, item = 'ink'` → `25`
- `prices = {'pen': 10}, item = 'cap'` → `0`

In [ ]:
def price_of(prices, item):
    """
    Return the price of item from the price dict, or 0 if it isn't there.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(price_of, [
    (({'pen': 10, 'ink': 25}, 'ink'), 25),
    (({'pen': 10}, 'cap'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check with `in` first, then index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`prices.get(item, 0)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check with `in` first, then index.

```python
def price_of(prices, item):
    if item in prices:
        return prices[item]
    else:
        return 0
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `prices.get(item, 0)`.

```python
def price_of(prices, item):
    return prices.get(item, 0)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check with `in` first, then index. That costs O(1) time and O(1) space."

"I can do better: `prices.get(item, 0)`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/safe-lookup)

<a id="q5"></a>
### Q5. Most Common Word

🟢 Warm-up · *max with key*

Return the most frequent word (lower-cased). On a tie, return the one that appears first.

**Examples**

- `text = 'Bob hit a ball the hit BALL flew'` → `'hit'`
- `text = 'one'` → `'one'`

In [ ]:
def most_common_word(text):
    """
    Return the most frequent word (lower-cased). On a tie, return the one that
    appears first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(most_common_word, [
    (('Bob hit a ball the hit BALL flew',), 'hit'),
    (('one',), 'one'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count every word with list.count and keep the best.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count once into a dict, then `max(counts, key=counts.get)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count every word with list.count and keep the best.

```python
def most_common_word(text):
    words = text.lower().split()
    best, best_count = None, 0
    for w in words:
        c = words.count(w)
        if c > best_count:
            best, best_count = w, c
    return best
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Count once into a dict, then `max(counts, key=counts.get)`.

```python
def most_common_word(text):
    counts = {}
    for w in text.lower().split():
        counts[w] = counts.get(w, 0) + 1
    return max(counts, key=counts.get)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count every word with list.count and keep the best. That costs O(n²) time and O(n) space."

"I can do better: Count once into a dict, then `max(counts, key=counts.get)`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/most-common-word)

<a id="q6"></a>
### Q6. Pass or Fail Report

🟢 Warm-up · *Dict comprehension*

Given name → marks, return name → 'pass' (marks ≥ 40) or 'fail'.

**Examples**

- `marks = {'ana': 72, 'ben': 35}` → `{'ana': 'pass', 'ben': 'fail'}`
- `marks = {}` → `{}`

In [ ]:
def report(marks):
    """
    Given name → marks, return name → 'pass' (marks ≥ 40) or 'fail'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(report, [
    (({'ana': 72, 'ben': 35},), {'ana': 'pass', 'ben': 'fail'}),
    (({},), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over keys and fill a new dict with if/else.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict comprehension with a conditional expression.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over keys and fill a new dict with if/else.

```python
def report(marks):
    result = {}
    for name in marks:
        if marks[name] >= 40:
            result[name] = "pass"
        else:
            result[name] = "fail"
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A dict comprehension with a conditional expression.

```python
def report(marks):
    return {name: "pass" if m >= 40 else "fail" for name, m in marks.items()}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over keys and fill a new dict with if/else. That costs O(n) time and O(n) space."

"I can do better: A dict comprehension with a conditional expression. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/pass-or-fail-report)

<a id="q7"></a>
### Q7. Election Winner

🟢 Warm-up · *Counting votes*

Each vote is a candidate's name. Return the candidate with the most votes (first to reach the top count wins a tie).

**Examples**

- `votes = ['ana', 'ben', 'ana', 'cy', 'ben', 'ana']` → `'ana'`
- `votes = ['x']` → `'x'`

In [ ]:
def winner(votes):
    """
    Each vote is a candidate's name. Return the candidate with the most votes
    (first to reach the top count wins a tie).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(winner, [
    ((['ana', 'ben', 'ana', 'cy', 'ben', 'ana'],), 'ana'),
    ((['x'],), 'x'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Make a list of candidates, count each one by scanning the votes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count in one pass with a dict, then take the max by count.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Make a list of candidates, count each one by scanning the votes.

```python
def winner(votes):
    candidates = []
    for v in votes:
        if v not in candidates:
            candidates.append(v)
    best, best_count = None, -1
    for c in candidates:
        n = votes.count(c)
        if n > best_count:
            best, best_count = c, n
    return best
```

**Cost:** O(n · c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Count in one pass with a dict, then take the max by count.

```python
def winner(votes):
    tally = {}
    for v in votes:
        tally[v] = tally.get(v, 0) + 1
    return max(tally, key=tally.get)
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Make a list of candidates, count each one by scanning the votes. That costs O(n · c) time and O(c) space."

"I can do better: Count in one pass with a dict, then take the max by count. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/election-winner)

<a id="q8"></a>
### Q8. Two Lists to a Dict

🟢 Warm-up · *dict(zip())*

Build a dict from a list of keys and a list of values of the same length.

**Examples**

- `keys = ['a', 'b'], values = [1, 2]` → `{'a': 1, 'b': 2}`
- `keys = [], values = []` → `{}`

In [ ]:
def to_dict(keys, values):
    """
    Build a dict from a list of keys and a list of values of the same length.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_dict, [
    ((['a', 'b'], [1, 2]), {'a': 1, 'b': 2}),
    (([], []), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`dict(zip(keys, values))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices.

```python
def to_dict(keys, values):
    result = {}
    for i in range(len(keys)):
        result[keys[i]] = values[i]
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `dict(zip(keys, values))`.

```python
def to_dict(keys, values):
    return dict(zip(keys, values))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices. That costs O(n) time and O(n) space."

"I can do better: `dict(zip(keys, values))`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/two-lists-to-a-dict)

<a id="q9"></a>
### Q9. Filter by Value

🟢 Warm-up · *Comprehension with a condition*

Return only the items whose value is at least `minimum`.

**Examples**

- `d = {'a': 5, 'b': 1, 'c': 8}, minimum = 5` → `{'a': 5, 'c': 8}`
- `d = {'a': 1}, minimum = 9` → `{}`

In [ ]:
def filter_values(d, minimum):
    """
    Return only the items whose value is at least minimum.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(filter_values, [
    (({'a': 5, 'b': 1, 'c': 8}, 5), {'a': 5, 'c': 8}),
    (({'a': 1}, 9), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and copy the items that pass.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`{k: v for k, v in d.items() if v >= minimum}`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and copy the items that pass.

```python
def filter_values(d, minimum):
    result = {}
    for k in d:
        if d[k] >= minimum:
            result[k] = d[k]
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `{k: v for k, v in d.items() if v >= minimum}`.

```python
def filter_values(d, minimum):
    return {k: v for k, v in d.items() if v >= minimum}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and copy the items that pass. That costs O(n) time and O(n) space."

"I can do better: `{k: v for k, v in d.items() if v >= minimum}`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/filter-by-value)

<a id="q10"></a>
### Q10. Key With the Largest Value

🟢 Warm-up · *max(d, key=d.get)*

Return the key whose value is largest (first one on ties).

**Examples**

- `d = {'a': 3, 'b': 9, 'c': 9}` → `'b'`
- `d = {'x': -1}` → `'x'`

In [ ]:
def key_of_max(d):
    """
    Return the key whose value is largest (first one on ties).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(key_of_max, [
    (({'a': 3, 'b': 9, 'c': 9},), 'b'),
    (({'x': -1},), 'x'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the items keeping the best key.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max(d, key=d.get)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over the items keeping the best key.

```python
def key_of_max(d):
    best = None
    for k in d:
        if best is None or d[k] > d[best]:
            best = k
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `max(d, key=d.get)`.

```python
def key_of_max(d):
    return max(d, key=d.get)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the items keeping the best key. That costs O(n) time and O(1) space."

"I can do better: `max(d, key=d.get)`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/key-with-the-largest-value)

<a id="q11"></a>
### Q11. Total of All Values

🟢 Warm-up · *values()*

Return the sum of all values in a dict of numbers.

**Examples**

- `d = {'a': 3, 'b': 4}` → `7`
- `d = {}` → `0`

In [ ]:
def total_values(d):
    """
    Return the sum of all values in a dict of numbers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(total_values, [
    (({'a': 3, 'b': 4},), 7),
    (({},), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over keys and add `d[k]`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sum(d.values())`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over keys and add `d[k]`.

```python
def total_values(d):
    total = 0
    for k in d:
        total += d[k]
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sum(d.values())`.

```python
def total_values(d):
    return sum(d.values())
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over keys and add `d[k]`. That costs O(n) time and O(1) space."

"I can do better: `sum(d.values())`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/total-of-all-values)

<a id="q12"></a>
### Q12. First Repeated Word

🟢 Warm-up · *Seen set / dict*

Return the first word that appears a second time, or None.

**Examples**

- `text = 'he had had quite enough'` → `'had'`
- `text = 'all different here'` → `None`

In [ ]:
def first_repeat(text):
    """
    Return the first word that appears a second time, or None.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_repeat, [
    (('he had had quite enough',), 'had'),
    (('all different here',), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each word, check whether it appears earlier in the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Remember words in a set as you go.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each word, check whether it appears earlier in the list.

```python
def first_repeat(text):
    words = text.split()
    for i in range(len(words)):
        if words[i] in words[:i]:
            return words[i]
    return None
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Remember words in a set as you go.

```python
def first_repeat(text):
    seen = set()
    for w in text.split():
        if w in seen:
            return w
        seen.add(w)
    return None
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each word, check whether it appears earlier in the list. That costs O(n²) time and O(n) space."

"I can do better: Remember words in a set as you go. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/first-repeated-word)

<a id="q13"></a>
### Q13. Total per Category

🟢 Warm-up · *Summing into a dict*

`items` is a list of (category, amount). Return category → total amount.

**Examples**

- `items = [('food', 5), ('bus', 2), ('food', 3)]` → `{'food': 8, 'bus': 2}`
- `items = []` → `{}`

In [ ]:
def per_category(items):
    """
    items is a list of (category, amount). Return category → total amount.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(per_category, [
    (([('food', 5), ('bus', 2), ('food', 3)],), {'food': 8, 'bus': 2}),
    (([],), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check for the key each time and branch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict(int)` starts every new total at 0.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check for the key each time and branch.

```python
def per_category(items):
    totals = {}
    for pair in items:
        cat = pair[0]
        amount = pair[1]
        if cat in totals:
            totals[cat] += amount
        else:
            totals[cat] = amount
    return totals
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `defaultdict(int)` starts every new total at 0.

```python
def per_category(items):
    from collections import defaultdict
    totals = defaultdict(int)
    for cat, amount in items:
        totals[cat] += amount
    return dict(totals)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check for the key each time and branch. That costs O(n) time and O(k) space."

"I can do better: `defaultdict(int)` starts every new total at 0. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/total-per-category)

<a id="q14"></a>
### Q14. Merge Dicts, Adding Values

🟡 Interview standard · *Merging*

Merge two dicts of item → quantity; when a key is in both, add the quantities.

**Examples**

- `a = {'a': 1, 'b': 2}, b = {'b': 3, 'c': 4}` → `{'a': 1, 'b': 5, 'c': 4}`
- `a = {}, b = {'x': 1}` → `{'x': 1}`

In [ ]:
def merge_add(a, b):
    """
    Merge two dicts of item → quantity; when a key is in both, add the
    quantities.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge_add, [
    (({'a': 1, 'b': 2}, {'b': 3, 'c': 4}), {'a': 1, 'b': 5, 'c': 4}),
    (({}, {'x': 1}), {'x': 1}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy the first dict, then for each key of the second check whether it exists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Copy, then `result[k] = result.get(k, 0) + v` — no branch needed. (`Counter(a) + Counter(b)` works too.)

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Copy the first dict, then for each key of the second check whether it exists.

```python
def merge_add(a, b):
    result = {}
    for k in a:
        result[k] = a[k]
    for k in b:
        if k in result:
            result[k] = result[k] + b[k]
        else:
            result[k] = b[k]
    return result
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Copy, then `result[k] = result.get(k, 0) + v` — no branch needed. (`Counter(a) + Counter(b)` works too.)

```python
def merge_add(a, b):
    result = dict(a)
    for k, v in b.items():
        result[k] = result.get(k, 0) + v
    return result
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy the first dict, then for each key of the second check whether it exists. That costs O(m + n) time and O(m + n) space."

"I can do better: Copy, then `result[k] = result.get(k, 0) + v` — no branch needed. (`Counter(a) + Counter(b)` works too.) That's O(m + n) time and O(m + n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/merge-dicts-adding-values)

<a id="q15"></a>
### Q15. Group Words by First Letter

🟡 Interview standard · *setdefault / defaultdict*

Return a dict mapping each first letter to the list of words starting with it, in original order.

**Examples**

- `words = ['apple', 'bat', 'avocado', 'ball']` → `{'a': ['apple', 'avocado'], 'b': ['bat', 'ball']}`
- `words = []` → `{}`

In [ ]:
def group_by_letter(words):
    """
    Return a dict mapping each first letter to the list of words starting with
    it, in original order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(group_by_letter, [
    ((['apple', 'bat', 'avocado', 'ball'],), {'a': ['apple', 'avocado'], 'b': ['bat', 'ball']}),
    (([],), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check if the key exists; create the list if not; then append.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`groups.setdefault(key, []).append(word)` in one line.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check if the key exists; create the list if not; then append.

```python
def group_by_letter(words):
    groups = {}
    for w in words:
        first = w[0]
        if first not in groups:
            groups[first] = []
        groups[first].append(w)
    return groups
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `groups.setdefault(key, []).append(word)` in one line.

```python
def group_by_letter(words):
    groups = {}
    for w in words:
        groups.setdefault(w[0], []).append(w)
    return groups
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check if the key exists; create the list if not; then append. That costs O(n) time and O(n) space."

"I can do better: `groups.setdefault(key, []).append(word)` in one line. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/group-words-by-first-letter)

<a id="q16"></a>
### Q16. Group Anagrams

🟡 Interview standard · *Tuple / sorted keys*

Group words that are anagrams of each other. Return the groups as lists, in order of first appearance.

**Examples**

- `words = ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']` → `[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]`
- `words = ['a']` → `[['a']]`

In [ ]:
def group_anagrams(words):
    """
    Group words that are anagrams of each other. Return the groups as lists, in
    order of first appearance.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(group_anagrams, [
    ((['eat', 'tea', 'tan', 'ate', 'nat', 'bat'],), [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]),
    ((['a'],), [['a']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare each word with the first word of every existing group by sorting both.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the sorted letters as a dict key; all anagrams share it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare each word with the first word of every existing group by sorting both.

```python
def group_anagrams(words):
    groups = []
    for w in words:
        for g in groups:
            if sorted(g[0]) == sorted(w):
                g.append(w)
                break
        else:
            groups.append([w])
    return groups
```

**Cost:** O(n² · k log k) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the sorted letters as a dict key; all anagrams share it.

```python
def group_anagrams(words):
    groups = {}
    for w in words:
        groups.setdefault("".join(sorted(w)), []).append(w)
    return list(groups.values())
```

**Cost:** O(n · k log k) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare each word with the first word of every existing group by sorting both. That costs O(n² · k log k) time and O(n) space."

"I can do better: Use the sorted letters as a dict key; all anagrams share it. That's O(n · k log k) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/group-anagrams)

<a id="q17"></a>
### Q17. Sort by Value

🟡 Interview standard · *sorted on items()*

Return the keys sorted by their values from highest to lowest (ties keep insertion order).

**Examples**

- `d = {'a': 3, 'b': 9, 'c': 5}` → `['b', 'c', 'a']`
- `d = {'x': 1, 'y': 1}` → `['x', 'y']`

In [ ]:
def keys_by_value(d):
    """
    Return the keys sorted by their values from highest to lowest (ties keep
    insertion order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(keys_by_value, [
    (({'a': 3, 'b': 9, 'c': 5},), ['b', 'c', 'a']),
    (({'x': 1, 'y': 1},), ['x', 'y']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly find and remove the key with the largest value.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted(d, key=d.get, reverse=True)` — Python's sort is stable, so ties keep their order.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Repeatedly find and remove the key with the largest value.

```python
def keys_by_value(d):
    remaining = dict(d)
    order = []
    while remaining:
        best = None
        for k in remaining:
            if best is None or remaining[k] > remaining[best]:
                best = k
        order.append(best)
        del remaining[best]
    return order
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sorted(d, key=d.get, reverse=True)` — Python's sort is stable, so ties keep their order.

```python
def keys_by_value(d):
    return sorted(d, key=d.get, reverse=True)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly find and remove the key with the largest value. That costs O(n²) time and O(n) space."

"I can do better: `sorted(d, key=d.get, reverse=True)` — Python's sort is stable, so ties keep their order. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/sort-by-value)

<a id="q18"></a>
### Q18. Read a Nested Setting

🟡 Interview standard · *Nested dicts*

`config` is a dict of dicts. Return `config[section][key]`, or `None` if either level is missing.

**Examples**

- `config = {'db': {'port': 5432}}, section = 'db', key = 'port'` → `5432`
- `config = {'db': {}}, section = 'db', key = 'host'` → `None`
- `config = {}, section = 'web', key = 'port'` → `None`

In [ ]:
def setting(config, section, key):
    """
    config is a dict of dicts. Return config[section][key], or None if either
    level is missing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(setting, [
    (({'db': {'port': 5432}}, 'db', 'port'), 5432),
    (({'db': {}}, 'db', 'host'), None),
    (({}, 'web', 'port'), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each level with `in` before indexing.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Chain `get` calls with an empty dict as the first default.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each level with `in` before indexing.

```python
def setting(config, section, key):
    if section in config:
        inner = config[section]
        if key in inner:
            return inner[key]
    return None
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Chain `get` calls with an empty dict as the first default.

```python
def setting(config, section, key):
    return config.get(section, {}).get(key)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each level with `in` before indexing. That costs O(1) time and O(1) space."

"I can do better: Chain `get` calls with an empty dict as the first default. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/read-a-nested-setting)

<a id="q19"></a>
### Q19. Roman to Integer

🟡 Interview standard · *Lookup table*

Convert a Roman numeral (I, V, X, L, C, D, M) to an integer, e.g. 'MCMXCIV' → 1994.

**Examples**

- `s = 'III'` → `3`
- `s = 'LVIII'` → `58`
- `s = 'MCMXCIV'` → `1994`

In [ ]:
def roman(s):
    """
    Convert a Roman numeral (I, V, X, L, C, D, M) to an integer, e.g. 'MCMXCIV'
    → 1994.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(roman, [
    (('III',), 3),
    (('LVIII',), 58),
    (('MCMXCIV',), 1994),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Handle every special pair (IV, IX, XL, XC, CD, CM) with if statements while scanning.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Look up values in a dict; subtract a symbol when a bigger one follows it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Handle every special pair (IV, IX, XL, XC, CD, CM) with if statements while scanning.

```python
def roman(s):
    total = 0
    i = 0
    pairs = {"IV": 4, "IX": 9, "XL": 40, "XC": 90, "CD": 400, "CM": 900}
    singles = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    while i < len(s):
        if s[i:i + 2] in pairs:
            total += pairs[s[i:i + 2]]
            i += 2
        else:
            total += singles[s[i]]
            i += 1
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Look up values in a dict; subtract a symbol when a bigger one follows it.

```python
def roman(s):
    value = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    total = 0
    for i, ch in enumerate(s):
        if i + 1 < len(s) and value[ch] < value[s[i + 1]]:
            total -= value[ch]
        else:
            total += value[ch]
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Handle every special pair (IV, IX, XL, XC, CD, CM) with if statements while scanning. That costs O(n) time and O(1) space."

"I can do better: Look up values in a dict; subtract a symbol when a bigger one follows it. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/roman-to-integer)

<a id="q20"></a>
### Q20. Word Pattern

🟡 Interview standard · *Two-way mapping*

Return True if the words follow the letter pattern exactly, e.g. 'abba' and 'dog cat cat dog' → True; each letter maps to one word and each word to one letter.

**Examples**

- `pattern = 'abba', text = 'dog cat cat dog'` → `True`
- `pattern = 'abba', text = 'dog cat cat fish'` → `False`
- `pattern = 'aaaa', text = 'dog cat cat dog'` → `False`

In [ ]:
def follows_pattern(pattern, text):
    """
    Return True if the words follow the letter pattern exactly, e.g. 'abba' and
    'dog cat cat dog' → True; each letter maps to one word and each word to one
    letter.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(follows_pattern, [
    (('abba', 'dog cat cat dog'), True),
    (('abba', 'dog cat cat fish'), False),
    (('aaaa', 'dog cat cat dog'), False),
    (('ab', 'dog dog'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every pair of positions: same letter must mean same word and vice versa.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep two dicts (letter → word, word → letter) and check them in one pass.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare every pair of positions: same letter must mean same word and vice versa.

```python
def follows_pattern(pattern, text):
    words = text.split()
    if len(words) != len(pattern):
        return False
    for i in range(len(pattern)):
        for j in range(len(pattern)):
            if (pattern[i] == pattern[j]) != (words[i] == words[j]):
                return False
    return True
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep two dicts (letter → word, word → letter) and check them in one pass.

```python
def follows_pattern(pattern, text):
    words = text.split()
    if len(words) != len(pattern):
        return False
    to_word, to_letter = {}, {}
    for p, w in zip(pattern, words):
        if to_word.setdefault(p, w) != w or to_letter.setdefault(w, p) != p:
            return False
    return True
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every pair of positions: same letter must mean same word and vice versa. That costs O(n²) time and O(1) space."

"I can do better: Keep two dicts (letter → word, word → letter) and check them in one pass. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/dictionaries/word-pattern)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What can be a dictionary key?</b></summary>

Any hashable object: numbers, strings, booleans, None, tuples of hashable items, frozensets. Lists, dicts and sets can't be keys because they are mutable.

</details>

<details>
<summary><b>2. d[key] vs d.get(key)?</b></summary>

`d[key]` raises KeyError if the key is missing; `d.get(key, default)` returns the default (None if not given) instead.

</details>

<details>
<summary><b>3. Are dictionaries ordered?</b></summary>

Yes, since Python 3.7 they preserve insertion order. They are still not sorted — use `sorted()` on keys or items for a sorted order.

</details>

<details>
<summary><b>4. What is the time complexity of dict lookup?</b></summary>

O(1) on average thanks to hashing; O(n) in the rare worst case where many keys collide.

</details>

<details>
<summary><b>5. setdefault vs defaultdict?</b></summary>

Both supply a starting value for missing keys. `setdefault` is a method you call each time with the default; `defaultdict` is a dict subclass that creates the default automatically for any missing key.

</details>

<details>
<summary><b>6. How do you merge two dictionaries?</b></summary>

`a | b` (Python 3.9+) or `{**a, **b}` create a new dict; `a.update(b)` or `a |= b` change a in place. On shared keys, b's values win.

</details>

<details>
<summary><b>7. Why are 1 and True the same dict key?</b></summary>

They are equal (`1 == True`) and have the same hash, so the dict treats them as one key.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Keys and values
- [ ] I can explain: Reading safely and removing
- [ ] I can explain: Looping over a dictionary
- [ ] I can explain: Counting and grouping
- [ ] I can explain: Dict comprehensions and sorting
- [ ] I can explain: Nested data and mappings

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Group Anagrams
- [ ] [Q17](#q17) Sort by Value
- [ ] [Q18](#q18) Read a Nested Setting
- [ ] [Q19](#q19) Roman to Integer
- [ ] [Q20](#q20) Word Pattern

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.